# PS S6E6 â€” Stellar Classification: Baseline

**Goal:** Establish a scored LB anchor.  
**Metric:** Balanced accuracy  
**Steps:**
1. Preprocessing + feature engineering (color indices from EDA)
2. Threshold rule baseline (redshift only â€” sanity floor)
3. LightGBM â€” stratified 5-fold CV with OOF score
4. Feature importance
5. Generate submission

## 1. Imports & Config

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb

from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
CFG = dict(
    n_folds      = 5,
    seed         = 42,
    n_estimators = 1000,
    early_stop   = 50,
    # Redshift thresholds for the rule-based sanity baseline
    star_thresh  = 0.15,   # redshift < this  â†’ STAR
    qso_thresh   = 1.00,   # redshift > this  â†’ QSO
)

LGBM_PARAMS = dict(
    n_estimators    = CFG['n_estimators'],
    learning_rate   = 0.05,
    num_leaves      = 127,
    min_child_samples = 50,
    subsample       = 0.8,
    colsample_bytree = 0.8,
    class_weight    = 'balanced',
    n_jobs          = -1,
    random_state    = CFG['seed'],
    verbose         = -1,
)

## 2. Load Data

In [ ]:
train = pd.read_csv('/kaggle/input/datasets/ekowannanindome/stellar-dataset/train.csv', index_col='id')
test  = pd.read_csv('/kaggle/input/datasets/ekowannanindome/stellar-dataset/test.csv',  index_col='id')

print(f'Train: {train.shape}  |  Test: {test.shape}')

## 3. Feature Engineering

In [ ]:
def add_color_indices(df):
    df = df.copy()
    df['u_g'] = df['u'] - df['g']
    df['g_r'] = df['g'] - df['r']
    df['r_i'] = df['r'] - df['i']
    df['i_z'] = df['i'] - df['z']
    df['u_r'] = df['u'] - df['r']
    df['g_i'] = df['g'] - df['i']
    df['g_z'] = df['g'] - df['z']
    return df

train = add_color_indices(train)
test  = add_color_indices(test)

COLOR_COLS = ['u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'g_i', 'g_z']
print('Color indices added:', COLOR_COLS)

## 4. Preprocessing

In [ ]:
CAT_COLS = ['spectral_type', 'galaxy_population']
NUM_COLS = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']
FEAT_COLS = NUM_COLS + COLOR_COLS + CAT_COLS
TARGET = 'class'

# Encode categoricals â€” fit on train, apply to both
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
train[CAT_COLS] = oe.fit_transform(train[CAT_COLS])
test[CAT_COLS]  = oe.transform(test[CAT_COLS])

# Encode target
le = LabelEncoder()
y = le.fit_transform(train[TARGET])
print('Class mapping:', dict(zip(le.classes_, le.transform(le.classes_))))

X      = train[FEAT_COLS]
X_test = test[FEAT_COLS]
print(f'\nFeature matrix: {X.shape}')

## 5. Threshold Baseline (Redshift Rule)

Before any ML: how well does a simple redshift threshold do?

From EDA:
- STAR median redshift: 0.057
- GALAXY median redshift: 0.482  
- QSO median redshift: 1.799

In [ ]:
def threshold_predict(redshift_series, star_thresh, qso_thresh):
    preds = pd.Series('GALAXY', index=redshift_series.index)
    preds[redshift_series < star_thresh] = 'STAR'
    preds[redshift_series > qso_thresh]  = 'QSO'
    return preds

rule_preds = threshold_predict(
    train['redshift'],
    CFG['star_thresh'],
    CFG['qso_thresh']
)

rule_score = balanced_accuracy_score(train[TARGET], rule_preds)
print(f'Threshold rule balanced accuracy: {rule_score:.5f}')
print(f'  (star_thresh={CFG["star_thresh"]}, qso_thresh={CFG["qso_thresh"]})')

In [ ]:
# Confusion matrix for the rule baseline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

classes_ordered = ['GALAXY', 'QSO', 'STAR']
cm = confusion_matrix(train[TARGET], rule_preds, labels=classes_ordered)
disp = ConfusionMatrixDisplay(cm, display_labels=classes_ordered)
disp.plot(cmap='Blues')
plt.title(f'Threshold Rule â€” Balanced Acc: {rule_score:.4f}')
plt.tight_layout()

## 6. LightGBM â€” Stratified 5-Fold CV

In [ ]:
skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])

oof_proba  = np.zeros((len(X), len(le.classes_)))
test_proba = np.zeros((len(X_test), len(le.classes_)))
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    model = lgb.LGBMClassifier(**LGBM_PARAMS)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(CFG['early_stop'], verbose=False),
            lgb.log_evaluation(200),
        ],
        categorical_feature=CAT_COLS,
    )

    val_proba = model.predict_proba(X_val)
    val_preds = le.inverse_transform(val_proba.argmax(axis=1))
    val_true  = le.inverse_transform(y_val)

    score = balanced_accuracy_score(val_true, val_preds)
    fold_scores.append(score)
    oof_proba[val_idx] = val_proba
    test_proba += model.predict_proba(X_test) / CFG['n_folds']

    print(f'  Fold {fold+1}/{CFG["n_folds"]} | best_iter={model.best_iteration_} | balanced_acc={score:.5f}')

print(f'\nCV mean:  {np.mean(fold_scores):.5f}')
print(f'CV std:   {np.std(fold_scores):.5f}')

In [ ]:
# OOF score (more reliable than mean of fold scores for imbalanced problems)
oof_preds_labels = le.inverse_transform(oof_proba.argmax(axis=1))
oof_true_labels  = le.inverse_transform(y)
oof_score = balanced_accuracy_score(oof_true_labels, oof_preds_labels)

print(f'OOF balanced accuracy: {oof_score:.5f}')
print(f'Threshold rule:        {rule_score:.5f}')
print(f'Gain over rule:        +{oof_score - rule_score:.5f}')

In [ ]:
# OOF confusion matrix
cm_oof = confusion_matrix(oof_true_labels, oof_preds_labels, labels=classes_ordered)
disp_oof = ConfusionMatrixDisplay(cm_oof, display_labels=classes_ordered)
disp_oof.plot(cmap='Blues')
plt.title(f'LightGBM OOF â€” Balanced Acc: {oof_score:.4f}')
plt.tight_layout()

## 7. Feature Importance

In [ ]:
# Use the last fold's model â€” importance is stable across folds for this dataset size
importance_df = pd.DataFrame({
    'feature':   X.columns,
    'importance': model.feature_importances_,
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
importance_df.plot.barh(x='feature', y='importance', ax=ax, legend=False)
ax.set_title('LightGBM Feature Importance (last fold)')
ax.invert_yaxis()
plt.tight_layout()

print(importance_df.to_string(index=False))

## 8. Per-Class Performance

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(oof_true_labels, oof_preds_labels, target_names=classes_ordered))

## 9. Generate Submission

In [ ]:
test_pred_labels = le.inverse_transform(test_proba.argmax(axis=1))

submission = pd.DataFrame({'id': test.index, 'class': test_pred_labels})
submission.to_csv('submission_lgbm_baseline.csv', index=False)

print(f'Submission shape: {submission.shape}')
print(submission['class'].value_counts())
submission.head(10)

In [ ]:
from IPython.display import FileLink, display

display(FileLink("submission_lgbm_baseline.csv"))

## 10. Results Summary

| Model | CV Balanced Acc | LB Score | Notes |
|---|---|---|---|
| Threshold rule (redshift) | 0.83187 | — | No ML, redshift only |
| LightGBM baseline | 0.96418 | **0.96509** | Default params, color indices,  |

**LB vs CV delta:** +0.00091 — no overfitting, CV is reliable.

**Hardest class:** STAR (precision 0.89, recall 0.96) — ~11% of predicted STARs are mislabeled.

**Top features:** redshift dominates; color indices (g_r, u_g, g_i) next.

**Next steps for Week 2:**
- Investigate STAR misclassifications (what are they being confused with?)
- Hyperparameter tuning with Optuna
- Try XGBoost and CatBoost for ensemble diversity
- Add interaction features (redshift × color index)
